# XGBoost Learning Notebook


This notebook shows how to use **XGBoost** for classification, regression, feature importance,
evaluation, and parameter tuning.

It focuses on the most used classes, methods, and workflows rather than every API entry.

## Jupyter setup

```bash
pip install xgboost scikit-learn pandas numpy matplotlib jupyter
jupyter notebook
```

## Imports

In [ ]:
import xgboost as xgb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer, fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, mean_squared_error, r2_score
print("xgboost version:", xgb.__version__)

## Classification with XGBClassifier

In [ ]:
data = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.2, random_state=42, stratify=data.target
)

clf = xgb.XGBClassifier(
    n_estimators=200, learning_rate=0.05, max_depth=4,
    subsample=0.9, colsample_bytree=0.9, eval_metric="logloss", random_state=42
)
clf.fit(X_train, y_train)
pred = clf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))

## Feature importance

In [ ]:
importances = pd.Series(clf.feature_importances_, index=data.feature_names).sort_values(ascending=False)
print(importances.head(10))
importances.head(10).plot(kind="bar", figsize=(8,4), title="Top XGBoost Importances")
plt.show()

## Cross-validation and tuning

In [ ]:
scores = cross_val_score(
    xgb.XGBClassifier(eval_metric="logloss", random_state=42),
    data.data, data.target, cv=5, scoring="accuracy"
)
print("CV scores:", scores)
print("Mean CV accuracy:", scores.mean())

grid = GridSearchCV(
    xgb.XGBClassifier(eval_metric="logloss", random_state=42),
    param_grid={"max_depth": [3, 4], "n_estimators": [100, 200]},
    cv=3, scoring="accuracy"
)
grid.fit(X_train, y_train)
print("Best params:", grid.best_params_)
print("Best score:", grid.best_score_)

## Regression with XGBRegressor

In [ ]:
housing = fetch_california_housing()
X_train, X_test, y_train, y_test = train_test_split(
    housing.data, housing.target, test_size=0.2, random_state=42
)

reg = xgb.XGBRegressor(
    n_estimators=200, learning_rate=0.05, max_depth=4,
    subsample=0.9, colsample_bytree=0.9, random_state=42
)
reg.fit(X_train, y_train)
pred = reg.predict(X_test)
print("RMSE:", mean_squared_error(y_test, pred, squared=False))
print("R2:", r2_score(y_test, pred))

## Useful XGBoost classes and methods

In [ ]:
reference = {
    "Models": ["XGBClassifier", "XGBRegressor", "DMatrix"],
    "Fit/predict": ["fit", "predict", "predict_proba", "score"],
    "Attributes": ["feature_importances_", "best_score", "best_iteration"],
    "Plotting": ["plot_importance", "plot_tree"],
    "Training params": ["n_estimators", "learning_rate", "max_depth", "subsample", "colsample_bytree"]
}
for k, v in reference.items():
    print("\n" + k + ":")
    print(", ".join(v))